# Ablation: training sequence length `k` (GPT-2 and LSTM on Ant-Dir and Walker-Param)

Companion to `return_graph_vis.ipynb`: same W&B fetch, the **same local CSV cache**
(`rl_results/{project}/raw/`), the same grid filling and EMA as the `*_interpolated_padded.xlsx`
workbooks, and the same paper style. Each panel overlays two baselines (GPT-2, LSTM), each trained on a
`k = 32` transition window (`--max_seq_len 32`) and on the full 200-step episode (`k = 200`).

Encoding (factorised, so one legend per property is enough): **color = model** (the paper palette:
GPT-2 green, LSTM blue), **line style = k**: the standard setting `k = 32` (every baseline in the main
results uses it) is solid, the full-episode ablation `k = 200` dashed, both in the model's color. Curves
are seed means only (`SHOW_STD = False`): with four curves per panel the std bands were indistinguishable,
so the caption should state the std elsewhere (e.g. the final-return table below). The two legends are split across the two
panels: the left panel lists the models, the right one lists `k` (neutral gray handles).

Axes: x = **Environment Steps** (episodes x episode length; MuJoCo episodes never terminate early, so
this equals the Learner's `info/env_steps`), ending exactly at the training budget with a labelled
last tick; y = **Avg. Return** (`eval/return` is the mean over the 64 evaluation episodes of one
evaluation round; "mean" is left to the caption's mean ± std across seeds).

Layout: the pair fills the text width (`\textwidth` = 5.5 in) and is exactly as tall as a main-figure row
(`PANEL_H` = 1.35 in). Both panels get the **same plot area** (`plot_area()`); only the left panel shows the
y label (as in the main figure), each file is only as large as its labels, and the files are included at
their native size, so the fonts stay at their final size (set in `paper_style.py`).

In [1]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator

try:
    display
except NameError:            # plain-python execution (smoke test); Jupyter defines display()
    display = print


from paper_style import set_paper_style   # shared style: font sizes live in paper_style.py only


set_paper_style()

## Config

`PANELS` has one entry per output PDF; `runs` maps each series `(model, k)` to its W&B run-name template
and the `h` / `l` the identity check expects. `seq32` = `--max_seq_len 32` (neither GPT-2 nor LSTM uses
STORE, so a contiguous BPTT window of 32 transitions per update), `seqfull` = the whole 200-step episode.

**Not a clean pair: Ant-Dir LSTM.** Its `k = 32` curve is the run used in the main figure
(`ant_lstm_h512_l1_rl2x512_s{0-3}`, h = 512, V100) so it matches the main results, while `k = 200` only exists at h = 256 (`ant_lstm_h256_l1_rl2x512_seqfull`).
Say so in the caption. (A matched h = 256 `k = 32` run, `ant_lstm_h256_l1_rl2x512_seq32`, also exists.)

In [2]:
ENTITY = "mate_research"
SEEDS = [0, 1, 2, 3, 4]

MODELS = {"GPT-2": "#009E73", "LSTM": "#0072B2"}     # fixed paper palette; order = legend order
SEQ_NAME = {"GPT-2": "gpt", "LSTM": "lstm"}          # expected config_seq.seq_model.name
VARIANTS = {                         # k label -> style; order = legend order, the first entry is drawn on top
    r"$k=32$":  dict(ls="-",  shade=0.0),     # standard setting of every baseline
    r"$k=200$": dict(ls="--", shade=0.0),     # ablation: full 200-step episode (same color; the dash carries k)
}
K32, K200 = VARIANTS
PANELS = {                           # out stem prefix -> panel; legend: "model" | "k" | None
    "ant-dir": dict(
        project="ant-dir", title="Ant-Dir", episode_len=200, legend="model", show_ylabel=True,
        runs={
            ("GPT-2", K32):  dict(template="ant_gpt_h512_l1_rl2x512_seq32_s{seed}_v2", h=512, l=1),
            ("GPT-2", K200): dict(template="ant_gpt_h512_l1_rl2x512_seqfull_s{seed}_v2", h=512, l=1),
            ("LSTM", K32):   dict(template="ant_lstm_h512_l1_rl2x512_s{seed}", h=512, l=1),             # main-figure run (s0-s3)
            ("LSTM", K200):  dict(template="ant_lstm_h256_l1_rl2x512_seqfull_s{seed}_v2", h=256, l=1),  # only h256 exists
        }),
    "walker-param": dict(
        project="walker-param", title="Walker-Param", episode_len=200, legend="k", show_ylabel=False,
        runs={
            ("GPT-2", K32):  dict(template="walker_gpt_h512_l1_rl2x512_seq32_s{seed}_v2", h=512, l=1),
            ("GPT-2", K200): dict(template="walker_gpt_h512_l1_rl2x512_seqfull_s{seed}_v2", h=512, l=1),
            ("LSTM", K32):   dict(template="walker_lstm_h512_l2_rl2x512_seq32_s{seed}_v2", h=512, l=2),
            ("LSTM", K200):  dict(template="walker_lstm_h512_l2_rl2x512_seqfull_s{seed}_v2", h=512, l=2),
        }),
}
DASHES = (3, 1.5)                    # same dash pattern as the reference bounds in the main figure

METRIC = "eval/return"
XLABEL = "Environment Steps"         # x = episodes x episode_len (fixed-length episodes -> equals info/env_steps)
YLABEL = "Avg. Return"               # eval/return = mean over the 64 eval episodes; shown on the left panel only
YLIM = None                          # e.g. (0, 1300); applied to every panel
LEGEND_IN_PANEL = dict(loc="lower right", ncol=2)   # ax.legend kwargs; one row keeps it short in a 1.35 in panel
LEGEND_COLOR = "#444444"             # "k" legend handles: neutral gray, the line style carries k

MAX_EPISODES = None                  # None -> inferred per panel from the data (printed)
HORIZON_EPISODES = None              # training budget in episodes (x-axis end); None -> last eval point rounded up to 1000
LAST_FRACTION = 0.10                 # xlsx Final_Return: mean EMA over the last 10% of training
EMA_FRACTION = 0.03                  # EMA time constant 1/(1-decay) = 3% of the panel's evaluation points, so
                                     # every env is smoothed over the same fraction of its training (user, 2026-09-23)
EMA_DECAY = 0.95                     # fixed decay, used only when EMA_FRACTION is None (the xlsx workbooks use 0.9)
MISSING_FRACTION_LIMIT = 0.10        # xlsx: exclude a run missing >10% of its expected grid
MAX_TRAILING_MISSING = 1             # at most 1 trailing point filled by hold-last, else excluded

# Panel layout: every panel of a figure gets the SAME plot area (axes box). A panel file is only as large
# as the labels it shows, so a row pays for the y label once (SHOW_YLABEL only on its leftmost panel) and
# every plot gets the saved space. Include the PDFs at their NATIVE size (no width=) with \hfill between.
TEXT_WIDTH = 5.5                     # ICLR \textwidth (in)
ROW_N, N_YLABEL = 2, 1               # panels per row / how many of them show a y label (the left one, as in the main figure)
ROW_GAP = 0.06                       # gap between neighbouring panels (in); the tick-label reserves add ~0.04
PANEL_H = 1.35                       # height of a panel WITH title and x label (in)  (= a main-figure row)
YTICK_RESERVE = "0000"               # widest y tick label in the figure: room reserved in every panel
XTICK_RESERVE_RIGHT = "50M"          # widest LAST x tick label (half of it overhangs the plot)
OUTER_PAD = 0.02                     # in
LINE_W = 0.9
BAND_ALPHA = 0.18
SHOW_STD = False                     # std bands off: with 4 curves per panel they were indistinguishable
LEGEND_EDGE = "#d9d9d9"              # legend box edge color; "black" for a hard frame, "none" for no edge
LEGEND_SHADOW = dict(size=1.5, alpha=0.35, layers=4)   # soft drop shadow (points, total darkness, blur steps); None -> none

RESULTS_DIR = Path("rl_results")     # cache: rl_results/{project}/raw/, shared with the other notebooks
FIG_DIR = Path("figures")
FORCE_REFRESH = False                # True -> ignore the CSV cache and re-download from W&B
UNFINISHED_MAX_AGE_H = 12            # a cached run that was NOT "finished" is re-downloaded once its cache is older

## W&B fetch with a local CSV cache

Identical to `final_return_vs_depth_vis.ipynb` (same files and meta format, so the notebooks and
`prefetch_wandb_cache.py` share downloads). Finished runs are never re-downloaded.

In [3]:
def _identity_check(meta, expected):
    problems = [f"{k}={meta.get(k)!r} (expected {v!r})" for k, v in expected.items()
                if meta.get(k) is not None and meta.get(k) != v]
    if problems:
        warnings.warn(f"[{meta['run_name']}] identity mismatch: " + "; ".join(problems))


def fetch_run(project, run_name, force=False):
    """Return (DataFrame[Step, Return], meta dict) or (None, meta) when the run is not found."""
    raw_dir = RESULTS_DIR / project / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    csv_path, meta_path = raw_dir / f"{run_name}.csv", raw_dir / f"{run_name}.meta.json"
    if meta_path.exists() and not force:
        meta = json.loads(meta_path.read_text())
        age_h = (time.time() - meta_path.stat().st_mtime) / 3600
        fresh = UNFINISHED_MAX_AGE_H is None or age_h < UNFINISHED_MAX_AGE_H
        if meta.get("state") == "missing" and fresh:
            return None, meta
        if csv_path.exists() and (meta.get("state") == "finished" or fresh):
            return pd.read_csv(csv_path), meta

    import wandb
    api = wandb.Api(timeout=120)
    runs = list(api.runs(f"{ENTITY}/{project}", filters={"display_name": run_name}))
    if not runs:
        warnings.warn(f"no W&B run named {run_name!r} in {ENTITY}/{project}")
        meta = {"run_name": run_name, "state": "missing", "checked_at": time.strftime("%Y-%m-%dT%H:%M:%S")}
        meta_path.write_text(json.dumps(meta, indent=2))
        if csv_path.exists():
            csv_path.unlink()
        return None, meta
    if len(runs) > 1:                      # prefer a finished run, newest among those
        runs.sort(key=lambda r: (r.state == "finished", str(r.created_at)))
        warnings.warn(f"{len(runs)} runs named {run_name!r}; using {runs[-1].id} (state={runs[-1].state})")
    run = runs[-1]
    cfg = run.config
    seq = cfg.get("config_seq", {}).get("seq_model", {})
    meta = {
        "run_name": run_name, "run_id": run.id, "state": run.state, "created_at": str(run.created_at),
        "eval_interval": cfg.get("config_env", {}).get("eval_interval"),
        "seq_name": seq.get("name"), "is_oracle": seq.get("is_oracle", False),
        "hidden_size": seq.get("hidden_size"), "n_layer": seq.get("n_layer"),
        "project_output": cfg.get("config_seq", {}).get("project_output"),
    }
    rows = [(row["_step"], row[METRIC]) for row in run.scan_history() if row.get(METRIC) is not None]
    df = pd.DataFrame(rows, columns=["Step", "Return"]).sort_values("Step").reset_index(drop=True)
    df.to_csv(csv_path, index=False)
    meta_path.write_text(json.dumps(meta, indent=2))
    return df, meta

## Grid filling and EMA (mirrors the workbook `Methodology` sheet)

Same as `return_graph_vis.ipynb`: expected grid `eval_interval, …, MAX_EPISODES`; > 10 % missing →
excluded; internal gaps linearly interpolated, leading gaps padded with the first value, ≤ 1
trailing gap held; then EMA with `ema[0] = x[0]`.

In [4]:
def expected_grid(eval_interval, max_episodes):
    return np.arange(eval_interval, int(max_episodes) + 1, eval_interval)


def fill_to_grid(steps, values, grid, missing_fraction_limit=MISSING_FRACTION_LIMIT,
                 max_trailing_missing=MAX_TRAILING_MISSING):
    """Returns (filled values on `grid` or None, status, info dict)."""
    steps = np.asarray(steps, dtype=np.int64)
    values = np.asarray(values, dtype=np.float64)
    on_grid = np.isin(steps, grid)
    n_off_grid = int((~on_grid).sum())
    obs = dict(zip(steps[on_grid], values[on_grid]))          # duplicates: last one wins
    present = np.array([s in obs for s in grid])
    n_expected, n_observed = len(grid), int(present.sum())
    info = dict(eval_points=n_observed, expected_eval_points=n_expected,
                n_missing=n_expected - n_observed, n_off_grid=n_off_grid,
                n_leading=0, n_trailing=0, n_internal=0, exclusion_reason="")
    if n_observed == 0:
        info["exclusion_reason"] = "no evaluations on the grid"
        return None, "excluded", info
    first, last = int(np.argmax(present)), int(len(grid) - 1 - np.argmax(present[::-1]))
    info["n_leading"], info["n_trailing"] = first, n_expected - 1 - last
    info["n_internal"] = info["n_missing"] - info["n_leading"] - info["n_trailing"]
    if info["n_missing"] / n_expected > missing_fraction_limit:
        info["exclusion_reason"] = f"missing fraction {info['n_missing'] / n_expected:.1%} > {missing_fraction_limit:.0%}"
        return None, "excluded", info
    if info["n_trailing"] > max_trailing_missing:
        info["exclusion_reason"] = f"{info['n_trailing']} trailing points missing > {max_trailing_missing}"
        return None, "excluded", info

    xs = grid[present]
    ys = np.array([obs[s] for s in xs])
    filled = np.empty(n_expected)
    filled[first:last + 1] = np.interp(grid[first:last + 1], xs, ys)   # internal linear interpolation
    filled[:first] = ys[0]                                            # leading: repeat first observed
    filled[last + 1:] = ys[-1]                                        # trailing: hold-last
    if info["n_leading"] > 0:
        status = "padded"
    elif info["n_trailing"] > 0:
        status = "tail_extended"
    elif info["n_internal"] > 0:
        status = "interpolated"
    else:
        status = "original"
    return filled, status, info


def ema_decay(n_points):
    """EMA decay whose time constant 1/(1-decay) is EMA_FRACTION of the n_points evaluations of a panel:
    Cheetah (195 evals) ~0.83, Hopper (390) ~0.91, ML (781) ~0.96, Ant/Walker (976) ~0.97."""
    if EMA_FRACTION is None:
        return EMA_DECAY
    return float(np.clip(1.0 - 1.0 / (EMA_FRACTION * n_points), 0.0, 0.999))


def ema(x, decay=EMA_DECAY):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = decay * out[t - 1] + (1.0 - decay) * x[t]
    return out

## Load every panel, print the per-seed quality table

In [5]:
def load_panel(panel, max_episodes=MAX_EPISODES, force=FORCE_REFRESH):
    raw = {(series, seed): fetch_run(panel["project"], spec["template"].format(seed=seed), force=force)
           for series, spec in panel["runs"].items() for seed in SEEDS}

    intervals = {m["eval_interval"] for df, m in raw.values() if df is not None}
    assert len(intervals) == 1, f"eval_interval differs across runs: {intervals}"
    eval_interval = int(intervals.pop())
    if max_episodes is None:
        last_steps = [int(df["Step"].max()) for df, _ in raw.values() if df is not None and len(df)]
        max_episodes = (max(last_steps) // eval_interval) * eval_interval
        print(f"[{panel['project']}] MAX_EPISODES inferred = {max_episodes} (eval_interval={eval_interval})")
    grid = expected_grid(eval_interval, max_episodes)

    results, rows = {}, []
    for series, spec in panel["runs"].items():
        model, k = series
        expected = dict(seq_name=SEQ_NAME[model], is_oracle=False, hidden_size=spec["h"], n_layer=spec["l"])
        curves = []
        for seed in SEEDS:
            df, meta = raw[(series, seed)]
            if df is None:
                rows.append(dict(model=model, k=k, seed=seed, run_name=meta["run_name"], state=meta["state"],
                                 status="missing", included=False, eval_points=0,
                                 expected=len(grid), n_missing=len(grid), reason="run not found"))
                continue
            _identity_check(meta, expected)
            filled, status, info = fill_to_grid(df["Step"].values, df["Return"].values, grid)
            if info["n_off_grid"]:
                warnings.warn(f"[{meta['run_name']}] dropped {info['n_off_grid']} off-grid evaluation rows")
            rows.append(dict(model=model, k=k, seed=seed, run_name=meta["run_name"], state=meta["state"],
                             status=status, included=filled is not None, eval_points=info["eval_points"],
                             expected=info["expected_eval_points"], n_missing=info["n_missing"],
                             reason=info["exclusion_reason"]))
            if filled is not None:
                curves.append(ema(filled, ema_decay(len(grid))))
        if not curves:
            warnings.warn(f"{model} {k}: no included seeds, it will not be drawn")
            continue
        per_seed = np.stack(curves)                     # (n_seeds, T)
        results[series] = dict(
            grid=grid, per_seed=per_seed, mean=per_seed.mean(0),
            std=per_seed.std(0, ddof=1) if len(per_seed) > 1 else np.zeros(len(grid)),
            n=len(per_seed),
        )
    return results, pd.DataFrame(rows)


loaded = {key: load_panel(panel) for key, panel in PANELS.items()}   # key -> (results, quality)
quality = pd.concat({key: q for key, (_r, q) in loaded.items()}, names=["panel", None])
with pd.option_context("display.max_rows", 200, "display.width", 200):
    display(quality)

wandb: Currently logged in as: himchan00 (piggene00) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[ant-dir] MAX_EPISODES inferred = 249856 (eval_interval=256)
[walker-param] MAX_EPISODES inferred = 249856 (eval_interval=256)


/tmp/ipykernel_497370/4131638616.py:26: UserWarning: no W&B run named 'ant_lstm_h512_l1_rl2x512_s4' in mate_research/ant-dir
  warnings.warn(f"no W&B run named {run_name!r} in {ENTITY}/{project}")


model        k  seed                                   run_name     state        status  included  eval_points  expected  n_missing         reason
panel                                                                                                                                                              
ant-dir      0   GPT-2   $k=32$     0        ant_gpt_h512_l1_rl2x512_seq32_s0_v2  finished  interpolated      True          974       976          2               
             1   GPT-2   $k=32$     1        ant_gpt_h512_l1_rl2x512_seq32_s1_v2  finished      original      True          976       976          0               
             2   GPT-2   $k=32$     2        ant_gpt_h512_l1_rl2x512_seq32_s2_v2  finished      original      True          976       976          0               
             3   GPT-2   $k=32$     3        ant_gpt_h512_l1_rl2x512_seq32_s3_v2  finished  interpolated      True          971       976          5               
             4   GPT-2   $k=32$     4        ant_gpt_h512_l1_rl2x512_seq32_s4_v2  finished  interpolated      True          975       976          1               
             5   GPT-2  $k=200$     0      ant_gpt_h512_l1_rl2x512_seqfull_s0_v2  finished  interpolated      True          970       976          6               
             6   GPT-2  $k=200$     1      ant_gpt_h512_l1_rl2x512_seqfull_s1_v2  finished      original      True          976       976          0               
             7   GPT-2  $k=200$     2      ant_gpt_h512_l1_rl2x512_seqfull_s2_v2  finished  interpolated      True          975       976          1               
             8   GPT-2  $k=200$     3      ant_gpt_h512_l1_rl2x512_seqfull_s3_v2  finished  interpolated      True          971       976          5               
             9   GPT-2  $k=200$     4      ant_gpt_h512_l1_rl2x512_seqfull_s4_v2  finished  interpolated      True          973       976          3               
             10   LSTM   $k=32$     0                ant_lstm_h512_l1_rl2x512_s0  finished      original      True          976       976          0               
             11   LSTM   $k=32$     1                ant_lstm_h512_l1_rl2x512_s1  finished  interpolated      True          975       976          1               
             12   LSTM   $k=32$     2                ant_lstm_h512_l1_rl2x512_s2  finished  interpolated      True          967       976          9               
             13   LSTM   $k=32$     3                ant_lstm_h512_l1_rl2x512_s3  finished  interpolated      True          968       976          8               
             14   LSTM   $k=32$     4                ant_lstm_h512_l1_rl2x512_s4   missing       missing     False            0       976        976  run not found
             15   LSTM  $k=200$     0     ant_lstm_h256_l1_rl2x512_seqfull_s0_v2  finished  interpolated      True          958       976         18               
             16   LSTM  $k=200$     1     ant_lstm_h256_l1_rl2x512_seqfull_s1_v2  finished  interpolated      True          967       976          9               
             17   LSTM  $k=200$     2     ant_lstm_h256_l1_rl2x512_seqfull_s2_v2  finished  interpolated      True          973       976          3               
             18   LSTM  $k=200$     3     ant_lstm_h256_l1_rl2x512_seqfull_s3_v2  finished  interpolated      True          963       976         13               
             19   LSTM  $k=200$     4     ant_lstm_h256_l1_rl2x512_seqfull_s4_v2  finished  interpolated      True          975       976          1               
walker-param 0   GPT-2   $k=32$     0     walker_gpt_h512_l1_rl2x512_seq32_s0_v2  finished  interpolated      True          975       976          1               
             1   GPT-2   $k=32$     1     walker_gpt_h512_l1_rl2x512_seq32_s1_v2  finished      original      True          976       976          0               
             2   GPT-2   $k=32$     2     walker_gpt_h512_l1_rl2x512_seq32_s2_v2 

## Panel

Mean ± std across seeds of the EMA-smoothed return, one legend inside each panel (`legend="model"` or
`"k"`). The x axis
runs from 0 to the training budget and its last tick is exactly the budget, so the total number of
training timesteps is always printed. Each panel
is saved as `figures/{key}_seq_len_ablation.pdf` (vector, Type-42 fonts) plus a PNG preview.

In [6]:
def _si_formatter(v, _pos):
    for div, suffix in ((1e9, "B"), (1e6, "M"), (1e3, "k")):
        if abs(v) >= div:
            return f"{v / div:g}{suffix}"
    return f"{v:g}"


def budget_tick_candidates(xmax):
    """Tick sets 0..xmax whose LAST tick is exactly xmax (the training budget), best first: a round step
    dividing xmax (densest first), then a round step whose last tick is replaced by / followed by xmax
    (e.g. T-Maze budgets = episodes x (T+1)), then [0, xmax/2, xmax], then [0, xmax]."""
    decade = 10.0 ** np.floor(np.log10(xmax))
    steps = [m * decade for m in (0.1, 0.2, 0.25, 0.5, 1, 2, 2.5, 5)]
    divides = [abs(xmax / s - round(xmax / s)) < 1e-6 for s in steps]
    for step, exact in zip(steps, divides):
        if exact and 3 <= round(xmax / step) <= 6:
            yield np.linspace(0, xmax, int(round(xmax / step)) + 1)
    for step, exact in zip(steps, divides):
        n = int(np.floor(xmax / step + 1e-9))
        if not exact and 2 <= n <= 6:
            ticks = list(np.arange(n + 1) * step)
            if xmax - ticks[-1] < 0.5 * step:
                ticks[-1] = xmax                               # too close to the budget to label both
            else:
                ticks.append(xmax)
            yield np.array(ticks)
    yield np.array([0, xmax / 2, xmax])
    yield np.array([0, xmax])


def set_budget_xticks(ax, xmax, min_gap_pt=1.0):
    """x axis 0..xmax with the densest candidate tick set whose labels do not collide (call after the
    labels/title are set: it draws the figure to measure the tick labels)."""
    ax.set_xlim(0, xmax)
    ax.xaxis.set_major_formatter(FuncFormatter(_si_formatter))
    fig = ax.figure
    gap = min_gap_pt * fig.dpi / 72
    for ticks in budget_tick_candidates(xmax):
        ax.set_xticks(ticks)
        fig.canvas.draw()
        boxes = sorted((t.get_window_extent() for t in ax.get_xticklabels() if t.get_text()), key=lambda b: b.x0)
        if all(a.x1 + gap <= b.x0 for a, b in zip(boxes, boxes[1:])):
            break
    ax.set_xlim(0, xmax)
    return ticks


def shade(color, t):
    """Mix `color` toward black by `t` (0 = unchanged): same hue, darker."""
    return matplotlib.colors.to_hex(np.array(matplotlib.colors.to_rgb(color)) * (1.0 - t))


def style_legend_frame(legend):
    """Thin light edge plus a soft drop shadow (stacked offset copies with decreasing alpha; stays vector in PDF)."""
    frame = legend.get_frame()
    frame.set_linewidth(0.4)
    if LEGEND_SHADOW:
        n, size, alpha = LEGEND_SHADOW["layers"], LEGEND_SHADOW["size"], LEGEND_SHADOW["alpha"]
        effects = [pe.SimplePatchShadow(offset=(size * k / n, -size * k / n), shadow_rgbFace="black",
                                        alpha=alpha / n) for k in range(n, 0, -1)]
        frame.set_path_effects(effects + [pe.Normal()])


def _text_extent(s, size, rotation=0, weight="normal"):
    """(width, height) in inches of `s` at `size` pt with the current rcParams (0 for an empty string)."""
    if not s:
        return 0.0, 0.0
    fig = plt.figure()
    t = fig.text(0, 0, s, fontsize=size, rotation=rotation, fontweight=weight)
    fig.canvas.draw()
    bb = t.get_window_extent()
    plt.close(fig)
    return bb.width / fig.dpi, bb.height / fig.dpi


def panel_margins(show_ylabel=True, show_xlabel=True, title=True):
    """Decoration space (left, right, bottom, top) in inches around the plot area. Tick labels always get the
    room of the widest expected label (YTICK_RESERVE, XTICK_RESERVE_RIGHT), so the plot area sits at the same
    place in every panel; only the y label, x label and title add space, and only where they are shown."""
    rc, pt = matplotlib.rcParams, 1 / 72
    ytick_w, ytick_h = _text_extent(YTICK_RESERVE, rc["ytick.labelsize"])
    xtick_h = _text_extent("0", rc["xtick.labelsize"])[1]
    left = OUTER_PAD + ytick_w + (rc["ytick.major.size"] + rc["ytick.major.pad"]) * pt
    if show_ylabel:
        left += _text_extent(YLABEL, rc["axes.labelsize"], rotation=90)[0] + rc["axes.labelpad"] * pt
    bottom = OUTER_PAD + xtick_h + (rc["xtick.major.size"] + rc["xtick.major.pad"]) * pt
    if show_xlabel:
        bottom += _text_extent(XLABEL, rc["axes.labelsize"])[1] + rc["axes.labelpad"] * pt
    top = OUTER_PAD + ytick_h / 2                              # the top y tick label overhangs the plot
    if title:
        title_h = _text_extent("Ag", rc["axes.titlesize"], weight=rc["axes.titleweight"])[1]
        top = max(top, OUTER_PAD + title_h + rc["axes.titlepad"] * pt)
    right = OUTER_PAD + _text_extent(XTICK_RESERVE_RIGHT, rc["xtick.labelsize"])[0] / 2
    return left, right, bottom, top


def plot_area():
    """(width, height) in inches of the plot area shared by every panel of the figure: ROW_N panels (N_YLABEL
    of them with a y label) plus ROW_GAP gaps fill TEXT_WIDTH, and a panel with title + x label is PANEL_H."""
    l1, r, b, t = panel_margins(show_ylabel=True)
    l0 = panel_margins(show_ylabel=False)[0]
    deco = N_YLABEL * l1 + (ROW_N - N_YLABEL) * l0 + ROW_N * r
    return (TEXT_WIDTH - (ROW_N - 1) * ROW_GAP - deco) / ROW_N, PANEL_H - b - t


def make_panel(show_ylabel=True, show_xlabel=True, title=True):
    """Figure whose plot area is exactly plot_area(); the file is only as large as the decorations it shows."""
    pw, ph = plot_area()
    left, right, bottom, top = panel_margins(show_ylabel, show_xlabel, title)
    w, h = left + pw + right, bottom + ph + top
    fig = plt.figure(figsize=(w, h))
    ax = fig.add_axes([left / w, bottom / h, pw / w, ph / h])
    return fig, ax


def check_panel_fits(fig, ax):
    """Warn when a label is larger than its reserve (it would be clipped at the file edge)."""
    fig.canvas.draw()
    bb, fb = ax.get_tightbbox(fig.canvas.get_renderer()), fig.bbox
    over = {"left": -bb.x0, "right": bb.x1 - fb.x1, "bottom": -bb.y0, "top": bb.y1 - fb.y1}
    over = {k: round(v / fig.dpi, 3) for k, v in over.items() if v > 0.5}
    if over:
        warnings.warn(f"labels exceed the panel by {over} in: widen YTICK_RESERVE / XTICK_RESERVE_RIGHT")
    return fig.get_size_inches()


def legend_handles(kind):
    if kind == "model":
        return [Line2D([], [], color=c, lw=LINE_W * 1.4, label=m) for m, c in MODELS.items()]
    return [Line2D([], [], color=LEGEND_COLOR, ls=v["ls"], lw=LINE_W * 1.4, label=k,
                   dashes=DASHES if v["ls"] != "-" else (None, None)) for k, v in VARIANTS.items()]


def plot_panel(results, panel, ylim=YLIM, out_stem=None):
    fig, ax = make_panel(show_ylabel=panel["show_ylabel"], title=bool(panel.get("title")))
    order = [(m, k) for k in reversed(VARIANTS) for m in MODELS]   # k = 32 drawn last (on top)
    for z, series in enumerate(order):
        if series not in results:
            continue
        r, (model, k) = results[series], series
        v = VARIANTS[k]
        color = shade(MODELS[model], v["shade"])
        x = r["grid"] * panel["episode_len"]
        if SHOW_STD:
            ax.fill_between(x, r["mean"] - r["std"], r["mean"] + r["std"],
                            color=color, alpha=BAND_ALPHA, lw=0, zorder=2 + 0.1 * z)
        ax.plot(x, r["mean"], color=color, ls=v["ls"], lw=LINE_W, zorder=3 + 0.1 * z,
                dashes=DASHES if v["ls"] != "-" else (None, None))
    last_eval = max(r["grid"][-1] for r in results.values())
    budget = (HORIZON_EPISODES or int(np.ceil(last_eval / 1000) * 1000)) * panel["episode_len"]
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    if ylim is not None:
        ax.set_ylim(*ylim)
    if panel.get("title"):
        ax.set_title(panel["title"])
    ax.set_xlabel(XLABEL)
    if panel["show_ylabel"]:
        ax.set_ylabel(YLABEL)
    ax.grid(True, ls="--", alpha=0.5)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    if LEGEND_IN_PANEL and panel.get("legend"):
        kw = dict(frameon=True, fancybox=False, edgecolor=LEGEND_EDGE, facecolor="white", framealpha=1.0,
                  handlelength=2.0, handletextpad=0.5, columnspacing=1.2, borderpad=0.35, borderaxespad=0.5)
        kw.update(LEGEND_IN_PANEL)
        legend = ax.legend(handles=legend_handles(panel["legend"]), **kw)
        legend.set_zorder(10)
        style_legend_frame(legend)
    set_budget_xticks(ax, budget)                                # last: measures the final tick labels

    size = check_panel_fits(fig, ax)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    stem = FIG_DIR / out_stem
    fig.savefig(stem.with_suffix(".pdf"))
    fig.savefig(stem.with_suffix(".png"), dpi=300)
    print(f"saved {stem.with_suffix('.pdf')} ({size[0]:.2f} x {size[1]:.2f} in, plot area "
          f"{plot_area()[0]:.2f} x {plot_area()[1]:.2f} in)")
    return fig, ax


for key, (results, _q) in loaded.items():
    fig, ax = plot_panel(results, PANELS[key], out_stem=f"{key}_seq_len_ablation")
    plt.show()

saved figures/ant-dir_seq_len_ablation.pdf (2.78 x 1.35 in, plot area 2.29 x 0.89 in)
saved figures/walker-param_seq_len_ablation.pdf (2.66 x 1.35 in, plot area 2.29 x 0.89 in)


## Final return (same definition as the workbooks)

Mean of the EMA curve over the last `LAST_FRACTION` of training, per seed, then mean ± std.

In [7]:
def final_return_table(results, last_fraction=LAST_FRACTION):
    rows = []
    for label, r in results.items():
        sel = r["grid"] >= r["grid"][-1] * (1 - last_fraction)
        for i, curve in enumerate(r["per_seed"]):
            rows.append(dict(model=label[0], k=label[1], seed_idx=i, final_return=curve[sel].mean()))
    return pd.DataFrame(rows)


for key, (results, _q) in loaded.items():
    print(key)
    display(final_return_table(results).groupby(["model", "k"], sort=False)["final_return"].agg(["mean", "std", "count"]))

ant-dir


mean        std  count
model k                                     
GPT-2 $k=32$   1073.619154  16.831043      5
      $k=200$  1055.380686  26.012548      5
LSTM  $k=32$   1116.840654  80.473707      4
      $k=200$   951.187975  41.860006      5

walker-param


mean        std  count
model k                                    
GPT-2 $k=32$   679.000984  92.660883      5
      $k=200$  828.699184  16.817116      5
LSTM  $k=32$   802.738104  79.370719      5
      $k=200$  639.854375  27.519402      5

## LaTeX

Appendix: the two panels side by side, filling the text width; the left legend explains the colors
(models), the right one the line styles (`k`).

```latex
\begin{figure}[t]
  \centering
  \includegraphics{figures/ant-dir_seq_len_ablation.pdf}\hfill
  \includegraphics{figures/walker-param_seq_len_ablation.pdf}
  \caption{... GPT-2 (green) and LSTM (blue) trained on $k=32$ windows (solid) or full episodes
    ($k=200$, dashed) ... Ant-Dir LSTM: $k=32$ is the main-table $h=512$ run, $k=200$ uses $h=256$ ...}
  \label{fig:seq-len-ablation}
\end{figure}
```

No `width=`: the panels are designed at their final size (same plot area, the files differ only by their
y tick labels), so nothing is rescaled and every text stays at the `paper_style.py` sizes, the same as the main figure.